In [ ]:
from src.environments import *

import time
import gymnasium as gym
import ipywidgets as widgets
from IPython.display import display

ENVIRONMENTS = [
    "safety_gridworlds/Base-v0",
    "safety_gridworlds/IslandNavigation-v0",
    "safety_gridworlds/Sokoban-v0",
    "safety_gridworlds/ConveyorBelt-v0",
]
output = widgets.Output()

pygame 2.6.1 (SDL 2.28.4, Python 3.11.11)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [ ]:
# Dropdown widget for environment selection
env_dropdown = widgets.Dropdown(
    options=ENVIRONMENTS,
    value=ENVIRONMENTS[1],
    description='Environment:',
)

# Button widget to launch environment
launch_button = widgets.Button(
    description='Launch',
    tooltip='Start PyGame loop with chosen environment'
)

display(env_dropdown, launch_button)

Dropdown(description='Environment:', index=1, options=('safety_gridworlds/Base-v0', 'safety_gridworlds/IslandN…

Button(description='Launch', style=ButtonStyle(), tooltip='Start PyGame loop with chosen environment')

In [ ]:
def launch_env(_):
    env_name = env_dropdown.value
    print(f"Launching environment: {env_name}")

    # Create the gym environment
    env = gym.make(env_name, render_mode="rgb_array")
    obs, info = env.reset()

    pygame.init()
    font = pygame.font.SysFont(None, 24)
    screen = pygame.display.set_mode((520, 520))
    pygame.display.set_caption(f"Interactive GridWorld: {env_name}")
    clock = pygame.time.Clock()

    action_names = ["RIGHT", "UP", "LEFT", "DOWN"]
    running = True
    total_reward = 0

    key_action_map = {
        pygame.K_RIGHT: 0,
        pygame.K_UP:    1,
        pygame.K_LEFT:  2,
        pygame.K_DOWN:  3,
    }

    while running:
        action = None
        reward = 0  # default for rendering text

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            elif event.type == pygame.KEYDOWN:
                if event.key in key_action_map:
                    action = key_action_map[event.key]

        if action is not None:
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            if terminated or truncated:
                obs, info = env.reset()
                txt = f"Resetting environment. Step reward={reward}, total_reward={total_reward}"
                print(txt)
                total_reward = 0

        # Render the frame from the env
        frame = env.render()  # shape: (height, width, 3)
        pygame_surface = pygame.surfarray.make_surface(
            np.transpose(frame, (1, 0, 2))
        )

        # Draw the environment
        screen.blit(pygame_surface, (0, 0))

        # Overlay text
        y_pos = 10
        texts = [
            f"Agent Position: {obs['agent']}" if 'agent' in obs else "Agent pos: N/A",
            f"Last Action: {action_names[action] if action is not None else 'None'}",
            f"Reward: {reward if action is not None else 'N/A'}",
            f"Total Reward (episode): {total_reward}",
            "Use Arrow Keys to Move, ESC to Quit",
        ]
        for text in texts:
            img = font.render(text, True, (0, 0, 0))
            screen.blit(img, (10, y_pos))
            y_pos += 30
        pygame.display.flip()
        clock.tick(10)

    env.close()
    pygame.quit()